In [ ]:
CATALOG = "demo_medallion"
BRONZE_SCHEMA = "bronze"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_status", StringType(), True)
])

order_data = [
    (1001, 101, "Laptop", 1, 75000.0, "COMPLETED"),
    (1002, 102, "Mouse", 2, 1500.0, "COMPLETED"),
    (1003, 103, "Keyboard", 1, 3000.0, "COMPLETED"),
    (1004, 104, "Monitor", 1, 18000.0, "CANCELLED"),
    (1005, 105, "Laptop", 1, 82000.0, "COMPLETED"),
    (1006, 101, "Headphones", 2, 5000.0, "COMPLETED"),

    # Duplicate
    (1003, 103, "Keyboard", 1, 3000.0, "COMPLETED"),

    # Invalid quantity
    (1007, 102, "Keyboard", -1, 3000.0, "COMPLETED"),

    # Invalid amount
    (1008, 106, "Monitor", 1, -5000.0, "COMPLETED")
]

df_orders = spark.createDataFrame(
    order_data,
    order_schema
)

display(df_orders)

In [ ]:
ORDER_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ORDER_TABLE} (
    order_id INT,
    customer_id INT,
    product STRING,
    quantity INT,
    amount DOUBLE,
    order_status STRING
)
USING DELTA
""")

In [ ]:
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(ORDER_TABLE)

print(f"Bronze table created/updated: {ORDER_TABLE}")